## Step 1: Install dependencies

Run the next cell once in this notebook's kernel. It installs PyMuPDF, NumPy, provider SDKs, and the other project requirements.

In [15]:
# %pip install -r requirements.txt

## Step 2: Load the project settings

Copy `.env.example` to `.env`, add whichever API keys you have, and put up to 10 PDFs in `data/pdfs/`. The code below reads settings only. It does not print keys or call a provider.

In [16]:
from pathlib import Path
import json
import sys
import yaml
from dotenv import load_dotenv

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "config.yaml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder so config.yaml can be found.")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CONFIG = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
SETTINGS = CONFIG["settings"]
MODEL_SETTINGS = CONFIG["models"]
PDF_DIR = PROJECT_ROOT / SETTINGS["pdf_dir"]
INDEX_DIR = PROJECT_ROOT / SETTINGS["index_dir"]
REPORT_DIR = PROJECT_ROOT / SETTINGS["report_dir"]
EVALUATION_FILE = PROJECT_ROOT / SETTINGS["evaluation_file"]
METRICS_FILE = PROJECT_ROOT / SETTINGS["metrics_file"]
PDF_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {PROJECT_ROOT}")
print(f"PDF folder: {PDF_DIR}")
for model_key, model in MODEL_SETTINGS.items():
    print(f"{model_key}: {model['model_id']} | ${model['price_per_million_tokens']}/1M tokens (estimate)")

Project folder: c:\Users\abhi0\OneDrive\Desktop\legalai
PDF folder: c:\Users\abhi0\OneDrive\Desktop\legalai\data\pdfs
openai-small: text-embedding-3-small | $0.02/1M tokens (estimate)
openai-large: text-embedding-3-large | $0.13/1M tokens (estimate)
voyage-4: voyage-4 | $0.06/1M tokens (estimate)


## Step 3: Extract pages and make shared chunks

PyMuPDF extracts selectable text page by page. Chunk metadata keeps the filename and 1-based page number. Every model uses this exact chunk list. Sparse pages are flagged because scanned PDFs may need OCR. This step is local and makes no API calls.

In [17]:
from pdf_processor import extract_and_chunk

chunks, pdf_warnings = extract_and_chunk(
    PDF_DIR,
    max_pdfs=SETTINGS["max_pdfs"],
    chunk_size=SETTINGS["chunk_size"],
    overlap=SETTINGS["chunk_overlap"],
    sparse_page_chars=SETTINGS["sparse_page_chars"],
)
for warning in pdf_warnings:
    print(f"WARNING: {warning}")
print(f"Prepared {len(chunks)} chunks from {len({chunk['filename'] for chunk in chunks})} PDF(s).")
print("Example metadata:", {key: chunks[0][key] for key in ("filename", "page", "chunk_number")})
print("Example text:", chunks[0]["text"][:400])

Prepared 12 chunks from 2 PDF(s).
Example metadata: {'filename': 'electrical_engineering.pdf', 'page': 1, 'chunk_number': 1}
Example text: ENGINEERING NOTES | Original sample text for retrieval testing 1 / 2 Electrical Engineering Fundamentals PART 1 | DC circuits and electrical power Voltage, current and resistance Electrical engineering studies circuits, energy conversion, signals and control. Voltage is electrical potential difference, measured in volts (V). Current is the rate of flow of electric charge, measured in amperes (A). 


## Step 4: Check provider keys

Only services with a configured key will be enabled. This creates SDK clients but does not send an embedding request. Keys are never printed.

In [18]:
from embedding_providers import create_configured_providers

providers, skipped_providers = create_configured_providers(
    MODEL_SETTINGS, SETTINGS["api_batch_size"]
)
for model_key in providers:
    print(f"Ready: {model_key} ({MODEL_SETTINGS[model_key]['model_id']})")
for model_key in skipped_providers:
    env_name = MODEL_SETTINGS[model_key]["api_key_env"]
    print(f"Skipped: {model_key}; {env_name} is not configured.")
if not providers:
    print("No providers are configured yet. Add API keys to .env before indexing.")

Ready: openai-small (text-embedding-3-small)
Ready: openai-large (text-embedding-3-large)
Ready: voyage-4 (voyage-4)


## Step 5: Embed and cache the chunks

**This cell can make paid API calls.** Each provider gets a separate vector cache. Unchanged chunks are reused; only new or changed chunks are sent. The reported costs are estimates at configured list prices and do not subtract free allowances or credits.

In [19]:
from retrieval import index_chunks
from evaluate import save_metrics

if METRICS_FILE.exists():
    metrics = json.loads(METRICS_FILE.read_text(encoding="utf-8"))
else:
    metrics = {"models": {}}
metrics.setdefault("models", {})
index_results = {}
for model_key, provider in providers.items():
    model = MODEL_SETTINGS[model_key]
    result = index_chunks(
        provider, model_key, chunks, INDEX_DIR, model["price_per_million_tokens"]
    )
    index_results[model_key] = result
    metrics["models"].setdefault(model_key, {}).update(result)
    print(
        f"{model_key}: {result['newly_embedded_chunks']} new, "
        f"{result['cached_chunks']} cached; {result['indexing_seconds']:.2f}s; "
        f"estimated cumulative index cost ${result['index_cost_usd_estimate_cumulative']:.6f}"
    )
save_metrics(METRICS_FILE, metrics)

openai-small: 0 new, 12 cached; 0.03s; estimated cumulative index cost $0.000037
openai-large: 0 new, 12 cached; 0.05s; estimated cumulative index cost $0.000240
voyage-4: 0 new, 12 cached; 0.01s; estimated cumulative index cost $0.000114


## Step 6: Ask a question and retrieve passages

**This cell sends your question to the selected embedding model and may incur a small API charge.** NumPy cosine similarity ranks only that model's saved vectors. A similarity score is not an accuracy percentage.

In [20]:
from retrieval import retrieve

selected_model = "openai-small"  # Choose openai-small, openai-large, or voyage-4.
question = "What is the heat-transfer rate through the plate?"
if selected_model not in providers:
    raise ValueError(f"{selected_model} is not configured; check its API key in .env.")

query_result = retrieve(
    providers[selected_model],
    selected_model,
    question,
    INDEX_DIR,
    top_k=5,
    price_per_million_tokens=MODEL_SETTINGS[selected_model]["price_per_million_tokens"],
)
for rank, passage in enumerate(query_result["matches"], start=1):
    print(
        f"\n[{rank}] {passage['filename']} | page {passage['page']} "
        f"| similarity {passage['similarity']:.4f}"
    )
    print(passage["text"])
print("\nSimilarity is a ranking score, not an accuracy percentage.")
print(
    f"Question embedding: {query_result['question_embedding_seconds']:.3f}s; "
    f"local search: {query_result['local_search_seconds']:.6f}s; "
    f"tokens: {query_result['query_tokens']} ({query_result['query_token_source']}); "
    f"estimated cost: ${query_result['query_cost_usd_estimate']:.8f}"
)


[1] mechanical_engineering.pdf | page 2 | similarity 0.5607
ENGINEERING NOTES | Original sample text for retrieval testing 2 / 2 Mechanical Engineering Fundamentals PART 2 | Heat transfer and machine reliability Three heat-transfer mechanisms Conduction transfers heat through matter, such as a metal housing. Convection transfers heat between a surface and a moving fluid, such as air passing over a motor. Radiation transfers energy through electromagnetic waves and does not require a material medium. Real equipment often experiences all three mechanisms together. Worked example: conduction through a plate For steady one-dimensional conduction through a flat plate, heat-transfer rate = k x A x temperature difference / thickness. Assume thermal conductivity k = 15 W/(m K), area A = 0.02 square metres, temperature difference = 20 K and thickness = 0.01 m. The calculated heat-transfer rate is 600 W. This idealized result neglects contact resistance and edge losses. Fatigue and vibration Fa

## Step 7: Evaluate labeled questions and export reports

Edit `evaluation_questions.json` and replace its example with questions and the correct supporting PDF pages. Example-only entries are ignored. Hit Rate@1 checks whether any correct page is ranked first; Hit Rate@5 checks the first five. Evaluation embeds each question with every configured model and may incur API charges. Without labels, accuracy is reported as unavailable, not guessed.

In [21]:
from evaluate import evaluate_provider, load_evaluation_questions, write_comparison_reports

questions, example_count = load_evaluation_questions(EVALUATION_FILE)
print(f"Active labeled questions: {len(questions)}; ignored examples: {example_count}")
for model_key, provider in providers.items():
    model = MODEL_SETTINGS[model_key]
    evaluation = evaluate_provider(
        provider, model_key, questions, INDEX_DIR, model["price_per_million_tokens"]
    )
    metrics["models"].setdefault(model_key, {}).update(evaluation)
    if evaluation["labeled_question_count"]:
        print(
            f"{model_key}: Hit Rate@1={evaluation['hit_rate_at_1']:.1%}; "
            f"Hit Rate@5={evaluation['hit_rate_at_5']:.1%}; "
            f"avg question embedding={evaluation['avg_question_embedding_seconds']:.3f}s; "
            f"avg local search={evaluation['avg_local_search_seconds']:.6f}s"
        )
    else:
        print(f"{model_key}: no labels, so hit rates are not measured.")
save_metrics(METRICS_FILE, metrics)

price_checked = max(model["price_checked"] for model in MODEL_SETTINGS.values())
csv_path, markdown_path = write_comparison_reports(
    REPORT_DIR, MODEL_SETTINGS, metrics, price_checked=price_checked
)
print(f"Saved {csv_path}")
print(f"Saved {markdown_path}")

Active labeled questions: 50; ignored examples: 0
openai-small: Hit Rate@1=98.0%; Hit Rate@5=100.0%; avg question embedding=0.473s; avg local search=0.000168s
openai-large: Hit Rate@1=98.0%; Hit Rate@5=100.0%; avg question embedding=0.675s; avg local search=0.000185s


RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

## What to review

Open `reports/comparison.md` for a readable summary or `reports/comparison.csv` for a spreadsheet. Reported cost is an estimate from configured list prices, before free allowances, credits, discounts, and billing adjustments. Index costs are cumulative in the local metrics file. Token counts are provider-reported when available; fallback estimates are marked. Local search time measures NumPy ranking only.

Prices were checked on **2026-10-15**: OpenAI `text-embedding-3-small` is $0.02 / 1M input tokens and `text-embedding-3-large` is $0.13 / 1M; Voyage `voyage-4` is $0.06 / 1M tokens and its pricing page lists an initial free allowance. Official references: [OpenAI embeddings guide](https://developers.openai.com/api/docs/guides/embeddings), [OpenAI pricing](https://developers.openai.com/api/docs/pricing), [Voyage embeddings guide](https://docs.voyageai.com/docs/embeddings), [Voyage Python client](https://docs.voyageai.com/docs/api-key-and-installation), [Voyage pricing](https://docs.voyageai.com/docs/pricing). Recheck prices before real use. For setup commands and the CLI alternative, see `README.md`.